In [1]:
landing_path = (
    "abfss://Fleet_Logistics_Engineering@onelake.dfs.fabric.microsoft.com/"
    "Fleet_Logistics_Lakehouse.Lakehouse/Files/Landing/delivery_events.csv"
)

StatementMeta(, dde9bd75-a195-40db-98f3-53d136ad33f9, 3, Finished, Available, Finished, False)

In [2]:
df_delivery = (
    spark.read
    .option("header", "true")
    .csv(landing_path)
)

display(df_delivery.limit(20))

df_delivery.printSchema()

print(f"Source records: {df_delivery.count()}")

StatementMeta(, dde9bd75-a195-40db-98f3-53d136ad33f9, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c87cccf1-8805-402a-a81a-efa26d59bbf0)

root
 |-- event_id: string (nullable = true)
 |-- load_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- facility_id: string (nullable = true)
 |-- scheduled_datetime: string (nullable = true)
 |-- actual_datetime: string (nullable = true)
 |-- detention_minutes: string (nullable = true)
 |-- on_time_flag: string (nullable = true)
 |-- location_city: string (nullable = true)
 |-- location_state: string (nullable = true)

Source records: 170820


In [4]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DateType,
    TimestampType
)

# Delivery Events Schema

delivery_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("load_id", StringType(), True),
    StructField("trip_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("facility_id", StringType(), True),
    StructField("scheduled_datetime", TimestampType(), True),
    StructField("actual_datetime", TimestampType(), True),
    StructField("detention_minutes", IntegerType(), True),
    StructField("on_time_flag", StringType(), True),
    StructField("location_city", StringType(), True),
    StructField("location_state", StringType(), True)
])

StatementMeta(, dde9bd75-a195-40db-98f3-53d136ad33f9, 6, Finished, Available, Finished, False)

In [5]:
landing_path = (
    "abfss://Fleet_Logistics_Engineering@onelake.dfs.fabric.microsoft.com/"
    "Fleet_Logistics_Lakehouse.Lakehouse/Files/Landing/delivery_events.csv"
)

df_delivery = (
    spark.read
    .option("header", "true")
    .schema(delivery_schema)
    .csv(landing_path)
)

display(df_delivery.limit(20))

df_delivery.printSchema()

print(f"Source records: {df_delivery.count()}")

StatementMeta(, dde9bd75-a195-40db-98f3-53d136ad33f9, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4d8fb9be-e8f7-442e-8705-fea651b5a497)

root
 |-- event_id: string (nullable = true)
 |-- load_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- facility_id: string (nullable = true)
 |-- scheduled_datetime: timestamp (nullable = true)
 |-- actual_datetime: timestamp (nullable = true)
 |-- detention_minutes: integer (nullable = true)
 |-- on_time_flag: string (nullable = true)
 |-- location_city: string (nullable = true)
 |-- location_state: string (nullable = true)

Source records: 170820


In [ ]:
# Validation

# 1. NULL primary key
null_event_ids = (
    df_delivery
    .filter(F.col("event_id").isNull())
    .count()
)

print(f"NULL event IDs: {null_event_ids}")


In [ ]:
# 2. Duplicate primary key
duplicate_event_ids = (
    df_delivery
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate event IDs: {duplicate_event_ids}")

In [ ]:
# 3. NULL scheduled datetime
null_scheduled_datetime = (
    df_delivery
    .filter(F.col("scheduled_datetime").isNull())
    .count()
)

print(f"NULL scheduled datetime: {null_scheduled_datetime}")

In [ ]:
# 4. NULL actual datetime
null_actual_datetime = (
    df_delivery
    .filter(F.col("actual_datetime").isNull())
    .count()
)

print(f"NULL actual datetime: {null_actual_datetime}")

In [6]:
# 5. Invalid detention minutes
invalid_detention_minutes = (
    df_delivery
    .filter(
        F.col("detention_minutes").isNull() |
        (F.col("detention_minutes") < 0)
    )
    .count()
)

print(f"Invalid detention minutes: {invalid_detention_minutes}")

StatementMeta(, dde9bd75-a195-40db-98f3-53d136ad33f9, 8, Finished, Available, Finished, False)

NULL event IDs: 0
Duplicate event IDs: 0
NULL scheduled datetime: 0
NULL actual datetime: 0
Invalid detention minutes: 0


In [7]:
# Referential Integrity  — Load
invalid_delivery_load_ids = (
    df_delivery
    .filter(F.col("load_id").isNotNull())
    .join(
        spark.table("bronze_loads").select("load_id"),
        on="load_id",
        how="left_anti"
    )
    .count()
)

print(f"Invalid non-NULL load IDs: {invalid_delivery_load_ids}")

StatementMeta(, dde9bd75-a195-40db-98f3-53d136ad33f9, 9, Finished, Available, Finished, False)

Invalid non-NULL load IDs: 0


In [8]:
# Referential Integrity — Trip
invalid_delivery_trip_ids = (
    df_delivery
    .filter(F.col("trip_id").isNotNull())
    .join(
        spark.table("bronze_trips").select("trip_id"),
        on="trip_id",
        how="left_anti"
    )
    .count()
)

print(f"Invalid non-NULL trip IDs: {invalid_delivery_trip_ids}")

StatementMeta(, dde9bd75-a195-40db-98f3-53d136ad33f9, 10, Finished, Available, Finished, False)

Invalid non-NULL trip IDs: 0


In [9]:
# Referential Integrity — Facility
invalid_delivery_facility_ids = (
    df_delivery
    .filter(F.col("facility_id").isNotNull())
    .join(
        spark.table("bronze_facilities").select("facility_id"),
        on="facility_id",
        how="left_anti"
    )
    .count()
)

print(
    f"Invalid non-NULL facility IDs: "
    f"{invalid_delivery_facility_ids}"
)

StatementMeta(, dde9bd75-a195-40db-98f3-53d136ad33f9, 11, Finished, Available, Finished, False)

Invalid non-NULL facility IDs: 0


In [10]:

if null_event_ids > 0:
    raise ValueError(
        "ETL failed: NULL event_id values detected."
    )

if duplicate_event_ids > 0:
    raise ValueError(
        "ETL failed: Duplicate event_id values detected."
    )

if null_scheduled_datetime > 0:
    raise ValueError(
        "ETL failed: NULL scheduled_datetime values detected."
    )

if invalid_detention_minutes > 0:
    raise ValueError(
        "ETL failed: Invalid detention minutes detected."
    )

if invalid_delivery_load_ids > 0:
    raise ValueError(
        "ETL failed: Delivery events contain "
        "load IDs not found in bronze_loads."
    )

if invalid_delivery_trip_ids > 0:
    raise ValueError(
        "ETL failed: Delivery events contain "
        "trip IDs not found in bronze_trips."
    )

if invalid_delivery_facility_ids > 0:
    raise ValueError(
        "ETL failed: Delivery events contain "
        "facility IDs not found in bronze_facilities."
    )

print(
    "Delivery Events data quality and "
    "referential-integrity validation passed."
)

StatementMeta(, dde9bd75-a195-40db-98f3-53d136ad33f9, 12, Finished, Available, Finished, False)

Delivery Events data quality and referential-integrity validation passed.


In [11]:
df_delivery_bronze = (
    df_delivery
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("delivery_events.csv"))
)

display(df_delivery_bronze.limit(10))

StatementMeta(, dde9bd75-a195-40db-98f3-53d136ad33f9, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bc662faf-4747-4c0c-a68f-39abe8cbd4de)

In [12]:
df_delivery_bronze.createOrReplaceTempView(
    "delivery_events_source"
)

StatementMeta(, dde9bd75-a195-40db-98f3-53d136ad33f9, 14, Finished, Available, Finished, False)

In [13]:
# Create Bronze Table
spark.sql("""
CREATE TABLE IF NOT EXISTS bronze_delivery_events (
    event_id STRING,
    load_id STRING,
    trip_id STRING,
    event_type STRING,
    facility_id STRING,
    scheduled_datetime TIMESTAMP,
    actual_datetime TIMESTAMP,
    detention_minutes INT,
    on_time_flag STRING,
    location_city STRING,
    location_state STRING,
    ingestion_timestamp TIMESTAMP,
    source_file STRING
)
""")

print("bronze_delivery_events table is ready.")

StatementMeta(, dde9bd75-a195-40db-98f3-53d136ad33f9, 15, Finished, Available, Finished, False)

bronze_delivery_events table is ready.


In [14]:
spark.sql("""
MERGE INTO bronze_delivery_events AS target

USING delivery_events_source AS source

ON target.event_id = source.event_id

WHEN MATCHED THEN
    UPDATE SET
        target.load_id = source.load_id,
        target.trip_id = source.trip_id,
        target.event_type = source.event_type,
        target.facility_id = source.facility_id,
        target.scheduled_datetime = source.scheduled_datetime,
        target.actual_datetime = source.actual_datetime,
        target.detention_minutes = source.detention_minutes,
        target.on_time_flag = source.on_time_flag,
        target.location_city = source.location_city,
        target.location_state = source.location_state,
        target.ingestion_timestamp = source.ingestion_timestamp,
        target.source_file = source.source_file

WHEN NOT MATCHED THEN
    INSERT (
        event_id,
        load_id,
        trip_id,
        event_type,
        facility_id,
        scheduled_datetime,
        actual_datetime,
        detention_minutes,
        on_time_flag,
        location_city,
        location_state,
        ingestion_timestamp,
        source_file
    )

    VALUES (
        source.event_id,
        source.load_id,
        source.trip_id,
        source.event_type,
        source.facility_id,
        source.scheduled_datetime,
        source.actual_datetime,
        source.detention_minutes,
        source.on_time_flag,
        source.location_city,
        source.location_state,
        source.ingestion_timestamp,
        source.source_file
    )
""")

print("Delivery Events Bronze MERGE completed successfully.")

StatementMeta(, dde9bd75-a195-40db-98f3-53d136ad33f9, 16, Finished, Available, Finished, False)

Delivery Events Bronze MERGE completed successfully.


In [15]:
bronze_delivery_count = (
    spark.table("bronze_delivery_events")
    .count()
)

print(f"Bronze delivery event records: {bronze_delivery_count}")

StatementMeta(, dde9bd75-a195-40db-98f3-53d136ad33f9, 17, Finished, Available, Finished, False)

Bronze delivery event records: 170820
